# Capítulo 3 — Métodos Paramétricos em Altas Dimensões

**Livro:** começa na página **51** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
set.seed(42)
options(digits = 4, repr.plot.width = 7, repr.plot.height = 4.5)
suppressPackageStartupMessages({ library(glmnet); library(rpart); library(randomForest); library(FNN); library(e1071) })
cat("Pronto.", R.version.string, "
")

## Busca pelo Melhor Subconjunto de Covariáveis · PDF p. 51

#### Exercício: Melhor subconjunto por AIC

Com `mtcars`, teste **todos** os subconjuntos das covariáveis `wt, hp, qsec, drat, disp` para prever `mpg` e mostre os 3 modelos de menor AIC. Quantos modelos foram ajustados?

In [ ]:
vars <- c("wt", "hp", "qsec", "drat", "disp")
subs <- unlist(lapply(1:5, function(k) combn(vars, k, simplify = FALSE)), recursive = FALSE)
aics <- sapply(subs, function(s) AIC(lm(reformulate(s, "mpg"), mtcars)))
cat("modelos ajustados:", length(subs), "\n")
head(data.frame(modelo = sapply(subs, paste, collapse = " + "), AIC = aics)[order(aics), ], 3)

## Regressão Stepwise · PDF p. 53

#### Exercício: Stepwise para frente

Faça a seleção *forward* com `step()`, partindo de `mpg ~ 1` e com escopo `mpg ~ wt + hp + qsec + drat + disp`. O resultado bate com o melhor subconjunto?

In [ ]:
nulo <- lm(mpg ~ 1, mtcars)
final <- step(nulo, scope = ~ wt + hp + qsec + drat + disp, direction = "forward", trace = 0)
formula(final)

## Lasso · PDF p. 55

#### Exercício: Lasso encontrando as variáveis certas

Simule n = 100, p = 50 covariáveis normais e y = 3x₁ − 2x₂ + 1,5x₃ + x₄ − x₅ + ruído. Ajuste `cv.glmnet(alpha = 1)` e mostre quais coeficientes ficaram diferentes de zero com `lambda.1se`.

In [ ]:
n <- 100; p <- 50
X <- matrix(rnorm(n * p), n, p); beta <- c(3, -2, 1.5, 1, -1, rep(0, p - 5))
y <- drop(X %*% beta + rnorm(n))
cv <- cv.glmnet(X, y, alpha = 1)
b <- coef(cv, s = "lambda.1se")[-1, 1]
which(b != 0); round(b[b != 0], 2)

## Regressão Ridge · PDF p. 58

#### Exercício: Caminho dos coeficientes

Com os mesmos dados simulados, ajuste ridge (`alpha = 0`) e lasso e desenhe o caminho dos coeficientes (`plot(glmnet(...), xvar = "lambda")`). Qual método zera coeficientes?

In [ ]:
n <- 100; p <- 50
X <- matrix(rnorm(n * p), n, p); y <- drop(X %*% c(3, -2, 1.5, 1, -1, rep(0, p - 5)) + rnorm(n))
par(mfrow = c(1, 2))
plot(glmnet(X, y, alpha = 0), xvar = "lambda", main = "Ridge")
plot(glmnet(X, y, alpha = 1), xvar = "lambda", main = "Lasso")
par(mfrow = c(1, 1))  # só o lasso leva coeficientes exatamente a zero

## Regressão ridge e lasso no R · PDF p. 61

#### Exercício: MQ × ridge × lasso no teste

Simule n = 60 e p = 40 (quase tantas variáveis quanto observações). Compare o EQM de teste (1.000 pontos) de mínimos quadrados, ridge e lasso (λ por validação cruzada).

In [ ]:
sim <- function(n, p = 40) { X <- matrix(rnorm(n * p), n, p); list(X = X, y = drop(X %*% c(2, -2, 1, rep(0, p - 3)) + rnorm(n, sd = 2))) }
tr <- sim(60); te <- sim(1000)
eqm <- function(prev) mean((te$y - prev)^2)
mq <- lm(tr$y ~ tr$X)
c(MQ = eqm(cbind(1, te$X) %*% coef(mq)),
  ridge = eqm(predict(cv.glmnet(tr$X, tr$y, alpha = 0), te$X, s = "lambda.min")),
  lasso = eqm(predict(cv.glmnet(tr$X, tr$y, alpha = 1), te$X, s = "lambda.min")))